In [ ]:
# 过滤警告'num_nodes'警告
import warnings


warnings.filterwarnings('ignore', 
    message="Unable to accurately infer 'num_nodes'",
    category=UserWarning,
    module='torch_geometric.data.storage')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_add_pool
from torch_geometric.nn.norm import BatchNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from kan import *
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')  # 使用方法.to(device, non_blocking=True) 

In [ ]:
# 环境诊断
print(torch.__version__)            # 确认torch版本
print(torch.cuda.is_available())    # 必须返回True
print(torch.cuda.get_device_name(0))# 应识别到3090

In [ ]:
# 1. 组合模型实现
class GAT_KAN(nn.Module):
    def __init__(self, gat_hyper_params, kan_hyper_params):
        super().__init__()
        
        # 初始化 GAT 层容器
        self.gat_layers = nn.ModuleList()  # 存储多个GAT层的容器
        self.batch_norms = nn.ModuleList()  # 存储BN层的容器
        
        # 输入特征维度（原子周期表序号 + 原子电荷）
        in_channels = 2
        
        # 构建隐藏层（贝叶斯优化决定层数和每层维度）
        for out_channels in gat_hyper_params['hidden_dims']:
            # 添加 GAT 层：多头注意力机制实现
            self.gat_layers.append(
                GATConv(
                    in_channels,            # 输入维度，每添加一层，in_channels更新一次与上一层的输出维度相匹配
                    out_channels,           # 输出维度（每个注意力头的维度）
                    heads=gat_hyper_params['heads']  # 注意力头数量
                )
            )
            
            # 添加对应的BatchNorm层
            self.batch_norms.append(
                BatchNorm(out_channels * gat_hyper_params['heads'])  # BN的输入维度与GAT输出维度一致
            )
            
            # 更新输入维度：多头注意力的输出维度 = 头数 × 每头维度
            # 例如：当 heads=4, out_channels=32 → 实际输出 4×32=128 维
            in_channels = out_channels * gat_hyper_params['heads']
        
        # 最终输出层配置
        self.gat_out = gat_hyper_params['out_dim']  # 输出维度（贝叶斯优化决定）
        
        # 添加最终输出层（使用单注意力头）
        self.gat_layers.append(GATConv(in_channels     # 输入来自最后一层隐藏层
                                       , self.gat_out  # 输出目标维度
                                       , heads=1       # 最终层只用单头注意力
                                      ))

        self.batch_norms.append(BatchNorm(self.gat_out))   # 输出层的BN
        
        # KAN的输入维度计算
        kan_input_dim = self.gat_out + 7  # 确保11与extra_features维度匹配

        # KAN的width结构
        kan_width = [kan_input_dim]  # 输入层
        for dim in kan_hyper_params['hidden_dims']:
            if isinstance(dim, list) and len(dim) == 2:
                kan_width.append(dim)  # 保持[加法节点数, 乘法节点数]结构
            else:
                kan_width.append([dim, 0])  # 普通层用0表示无乘法节点
        kan_width.append(1)  # 输出层

        # mult_arity结构
        kan_mult_arity = [tuple()]  # 输入层无乘法节点
        for arity in kan_hyper_params['mult_arity']:
            kan_mult_arity.append(arity)  # 保持每层的元数组
        kan_mult_arity.append(tuple())  # 输出层无乘法节点

        self.kan = MultKAN(width=kan_width
                           , mult_arity=kan_mult_arity
                           , k=kan_hyper_params['k']
                           , grid=kan_hyper_params['grid']
                           , seed=666
                           , device=device
                          )
        self.kan.speed()
        
    def forward(self, x, edge_index, batch, extra_features):
        # GAT处理
        for gat_layer, bn_layer in zip(self.gat_layers[:-1], self.batch_norms[:-1]):
            x = gat_layer(x, edge_index) # GAT计算
            x = bn_layer(x)              # BN归一化
            x = F.elu(x)                 # 激活函数
            x = F.dropout(x, p=0.5, training=self.training) # dropout
        x = self.gat_layers[-1](x, edge_index)
        x = self.batch_norms[-1](x)
        
        # 图池化
        pooled = global_mean_pool(x, batch)
        
        # 特征拼接
        combined = torch.cat([pooled, extra_features], dim=1)
        
        # KAN处理
        return self.kan(combined)

In [ ]:
def objective(trial):
    # GAT参数搜索空间
    gat_hyper_params = {
                'hidden_dims': [
                    trial.suggest_categorical(f'gat_hidden_{i}', [8, 16, 32, 64, 96, 128])
                    for i in range(trial.suggest_int('gat_layers', 1, 3))]
                , 'heads': trial.suggest_int('gat_heads', 1, 4)
                , 'out_dim': trial.suggest_int('gat_out', 2, 5)
                # , 'out_dim': 2
                }
    
    # KAN参数生成逻辑
    kan_hidden_dims = []
    kan_mult_arity = []
    
    for i in range(trial.suggest_int('kan_n_layers', 1, 4)):
        layer_type = trial.suggest_categorical(f'kan_layer_type_{i}', ['mixed', 'normal'])
        neurons = trial.suggest_int(f'kan_neurons_{i}', 1, 10)
        
        if layer_type == 'mixed':
            mult_nodes = trial.suggest_int(f'kan_mult_nodes_{i}', 1, 10)
            kan_hidden_dims.append([neurons, mult_nodes])
            
            # 为每个乘法节点生成元数
            layer_arity = tuple(
                trial.suggest_int(f'kan_arity_{i}_{j}', 2, 10)
                for j in range(mult_nodes)
            )
            kan_mult_arity.append(layer_arity)
        else:
            kan_hidden_dims.append([neurons, 0])
            kan_mult_arity.append(tuple())  # 普通层保持空元组

    kan_hyper_params = {
        'hidden_dims': kan_hidden_dims,
        'mult_arity': kan_mult_arity,
        'k': trial.suggest_int('kan_k', 3, 8),
        'grid': trial.suggest_int('kan_grid', 3, 8)
    }
    
    # 模型初始化
    model = GAT_KAN(gat_hyper_params, kan_hyper_params).to(device)
    
    # 训练验证流程
    optimizer = torch.optim.AdamW(model.parameters()
                                  , lr=trial.suggest_float('lr', 0.001, 0.01, log=True)
                                  # , lr=0.001
                                 )
    
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer
    #                                                        , T_max=30
    #                                                       )
    
    # 训练和验证代码
    # 损失函数
    criterion = nn.MSELoss().to(device) 

    num_epochs = 300
    # 训练循环
    model.train()
    for epoch in range(num_epochs):
        for data in train_loader:
            # data = data.to(device)
            optimizer.zero_grad()
            output1 = model(data.atom_types_chg
                            , data.edge_index
                            , data.batch
                            , data.experiment_condition_tensor)
            loss = criterion(output1, data.extraction_rate.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for val_data in val_loader:
            output2 = model(val_data.atom_types_chg
                            , val_data.edge_index
                            , val_data.batch
                            , val_data.experiment_condition_tensor)
            val_loss = criterion(output2, val_data.extraction_rate.float())
            val_loss += val_loss.item()
    
    return val_loss


In [ ]:
# 加载数据集
from pathlib import Path
# Run from the repository root or this notebook's directory.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'environment.yml').is_file() else Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / '02_datasets' / 'kare_gat_dataset.pt'
dataset = torch.load(DATASET_PATH
                     , map_location=device)

train_set, val_set = train_test_split(dataset
                                      , test_size=0.2   # 测试集比例 
                                      , random_state=666  # 随机种子（确保结果可复现）
                                      , shuffle=True    # 是否打乱数据（默认True）
                                     )

print(f'训练集大小：{len(train_set)}         验证集大小：{len(val_set)}')
print(train_set[0])

In [ ]:
train_loader = DataLoader(train_set
                          , batch_size=4080
                          , shuffle=True
                          # , num_workers=0  # 默认是0，多进程无法共享GPU数据
                          # , pin_memory=False  # 数据已在GPU，无需锁页内存
                         )

val_loader = DataLoader(val_set
                          , batch_size=1020
                          , shuffle=True
                          # , num_workers=0  # 默认是0，多进程无法共享GPU数据
                          # , pin_memory=False  # 数据已在GPU，无需锁页内存
                       )

In [ ]:
# 贝叶斯优化配置
study = optuna.create_study(direction='minimize'
                            , sampler=TPESampler(n_startup_trials=300)
                            , pruner=optuna.pruners.HyperbandPruner(min_resource=100
                                                                    ,reduction_factor=2)
                           )
study.optimize(objective
               , n_trials=3000
               , show_progress_bar=True)

In [ ]:
# loss最低的超参数
best_params = study.best_params
best_params

In [ ]:
best_value = study.best_value
print(f"最佳目标值：{best_value}")

In [ ]:
# 保存全部超参数优化的结果
study.trials_dataframe().to_csv(PROJECT_ROOT / '03_optimization' / 'optimization_results.csv', index=False)